In [1]:
import ee
import geemap
import sys
import os
sys.path.append(os.path.abspath("../"))
from dotenv import load_dotenv

# === Inicializar entorno ===
load_dotenv()
ee.Initialize(project=os.getenv('EE_PROJECT_ID'))

from compactacion_superficial.zona import get_zona
zona = get_zona()

In [2]:
Map = geemap.Map()
Map.centerObject(zona, 12)
Map.addLayer(zona, {'color': 'red'}, 'Zona La Joya')
Map

Map(center=[-16.72496480420877, -71.89237250000009], controls=(WidgetControl(options=['position', 'transparent…

In [3]:
from compactacion_superficial.ndmi import comparar_ndmi

ndmi_2019, ndmi_2023, cambio_ndmi = comparar_ndmi(
    zona,
    '2019-05-01', '2019-10-31',
    '2023-05-01', '2023-10-31'
)

Map.addLayer(ndmi_2019, {
    'min': -1, 'max': 1,
    'palette': ['brown', 'white', 'green']
}, 'NDMI 2019')

Map.addLayer(ndmi_2023, {
    'min': -1, 'max': 1,
    'palette': ['brown', 'white', 'green']
}, 'NDMI 2023')

Map.addLayer(cambio_ndmi, {
    'min': -0.5, 'max': 0.5,
    'palette': ['red', 'white', 'blue']
}, 'Cambio NDMI')

Map

Map(bottom=574010.0, center=[-16.72496480420877, -71.89237250000009], controls=(WidgetControl(options=['positi…

In [4]:
from compactacion_superficial.lst import comparar_lst

lst_2019, lst_2023, cambio_lst = comparar_lst(
    zona,
    '2019-05-01', '2019-10-31',
    '2023-05-01', '2023-10-31'
)

Map.addLayer(lst_2019, {'min': 15, 'max': 45, 'palette': ['blue', 'white', 'red']}, 'LST 2019')
Map.addLayer(lst_2023, {'min': 15, 'max': 45, 'palette': ['blue', 'white', 'red']}, 'LST 2023')
Map.addLayer(cambio_lst, {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']}, 'Cambio LST')
Map

Map(bottom=574010.0, center=[-16.72496480420877, -71.89237250000009], controls=(WidgetControl(options=['positi…

In [5]:
from compactacion_superficial.clasificacion import clasificacion_kmeans

clasificacion_img = clasificacion_kmeans(ndmi_2023, lst_2023, zona, n_clusters=4)

Map.addLayer(clasificacion_img.randomVisualizer(), {}, 'Clasificación KMeans')
Map

Map(bottom=574010.0, center=[-16.72496480420877, -71.89237250000009], controls=(WidgetControl(options=['positi…

In [6]:
from compactacion_superficial.tendencia import tendencia_ndmi
from compactacion_superficial.tendencia import contar_imagenes_anuales

for año in range(2019, 2024):
    print(f"Año {año}: {contar_imagenes_anuales(zona, año)} imágenes")

tendencia_img = tendencia_ndmi(zona)

stats = tendencia_img.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=zona,
    scale=30,
    maxPixels=1e9
)
print('Tendencia NDMI - min/max:', stats.getInfo())

tendencia_filtrada = tendencia_img.updateMask(tendencia_img.abs().gt(0.001))
Map.addLayer(tendencia_filtrada, {
    'min': -0.005, 'max': 0.005,
    'palette': ['red', 'white', 'green']
}, 'Tendencia NDMI 2019–2023 (solo cambios)')
Map


Año 2019: 75 imágenes
Año 2020: 74 imágenes
Año 2021: 66 imágenes
Año 2022: 109 imágenes
Año 2023: 88 imágenes
Tendencia NDMI - min/max: {'Tendencia_NDMI_max': 0.2144918441772461, 'Tendencia_NDMI_min': -0.1607998266816139}


Map(bottom=574010.0, center=[-16.72496480420877, -71.89237250000009], controls=(WidgetControl(options=['positi…